In [0]:
import pandas as pd
import numpy as np
import re
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType, BooleanType

# Load INFORM Severity and Complexity data using pandas
print("Loading INFORM severity and complexity data...")

# Read INFORM Severity CSV with pandas (skip title row)
inform_severity_pd = pd.read_csv("/Workspace/Users/maximilian.bach8@gmail.com/severity_csvs/INFORM_Severity_country.csv", skiprows=[0])

# Extract relevant columns (skip the header rows)
inform_severity_clean = inform_severity_pd[
    (inform_severity_pd["COUNTRY"].notna()) & 
    (inform_severity_pd["COUNTRY"] != "COUNTRY") &
    (inform_severity_pd["ISO3"] != "(a-z)")
][["ISO3", "INFORM Severity Index"]].copy()

inform_severity_clean.columns = ["iso3", "inform_severity_index"]
inform_severity_clean["inform_severity_index"] = pd.to_numeric(inform_severity_clean["inform_severity_index"], errors="coerce")

# DEDUPLICATE: Keep first occurrence of each country
inform_severity_clean = inform_severity_clean.drop_duplicates(subset=["iso3"], keep="first")

# Read Complexity CSV with pandas (use row 1 as header, skip MAX/MIN rows)
complexity_pd = pd.read_csv("/Workspace/Users/maximilian.bach8@gmail.com/severity_csvs/Complexity_of_the_crisis.csv", header=1, skiprows=[2,3])

# Rename column 4 (Unnamed: 4) to Iso3
complexity_pd = complexity_pd.rename(columns={"Unnamed: 4": "Iso3"})

# Extract complexity scores (skip header rows)
complexity_clean = complexity_pd[
    (complexity_pd["Iso3"].notna()) & 
    (complexity_pd["Iso3"] != "Iso3") &
    (complexity_pd["Iso3"] != "")
][["Iso3", "Operating environment"]].copy()

complexity_clean.columns = ["iso3", "complexity_operating_environment"]
complexity_clean["complexity_operating_environment"] = pd.to_numeric(complexity_clean["complexity_operating_environment"], errors="coerce")

# DEDUPLICATE: Keep first occurrence of each country
complexity_clean = complexity_clean.drop_duplicates(subset=["iso3"], keep="first")

print(f"  - INFORM severity data: {len(inform_severity_clean)} countries")
print(f"  - Complexity data: {len(complexity_clean)} countries\n")

# Get parameter values
crisis_category_full = dbutils.widgets.get("crisis_category")
include_temporal = dbutils.widgets.get("include_temporal_factor")
demographic_filter = dbutils.widgets.get("demographic_filter")

# Extract the code from the full name
crisis_category_match = re.search(r'\(([^)]+)\)$', crisis_category_full)
if crisis_category_match:
    crisis_category = crisis_category_match.group(1)
else:
    crisis_category = crisis_category_full

print(f"Selected: {crisis_category_full}")
print(f"Using code: {crisis_category}")

# Calculate demographic proportions per country
if demographic_filter != "All":
    if demographic_filter == "Children":
        demo_filter = "Age_range IN ('0-4', '5-9', '10-14', '15-19')"
    elif demographic_filter == "Men":
        demo_filter = "Gender = 'm' AND Age_range NOT IN ('0-4', '5-9', '10-14', '15-19', 'all')"
    elif demographic_filter == "Women":
        demo_filter = "Gender = 'f' AND Age_range NOT IN ('0-4', '5-9', '10-14', '15-19', 'all')"
    
    demographic_pop = spark.sql(f"""
    SELECT 
        ISO3 as iso3,
        SUM(Population) as demographic_population
    FROM workspace.unochadatasets.cod_population_admin_0
    WHERE {demo_filter}
    GROUP BY ISO3
    """)
    
    total_pop = spark.sql("""
    SELECT 
        ISO3 as iso3,
        SUM(Population) as total_population
    FROM workspace.unochadatasets.cod_population_admin_0
    WHERE Age_range = 'all' AND Gender = 'all'
    GROUP BY ISO3
    """)
    
    demo_proportions_spark = demographic_pop.join(total_pop, "iso3", "inner")
    demo_proportions = demo_proportions_spark.toPandas()
    demo_proportions["demographic_proportion"] = (
        demo_proportions["demographic_population"] / demo_proportions["total_population"]
    )
    demo_proportions = demo_proportions[["iso3", "demographic_proportion"]]
else:
    demo_proportions = pd.DataFrame(columns=["iso3", "demographic_proportion"])

# Load reach_ratio and total_population from actual HNO data (for context only)
hno_reach_spark = spark.sql(f"""
SELECT
  `Country ISO3` AS iso3,
  SUM(CAST(`Population` AS BIGINT)) AS total_population,
  (SUM(CAST(`Reached` AS DOUBLE)) / NULLIF(SUM(CAST(`Targeted` AS DOUBLE)), 0)) AS reach_ratio
FROM workspace.unochadatasets.hpc_hno_2025
WHERE `Cluster` = '{crisis_category}'
  AND (`Category` IS NULL OR TRIM(`Category`) = '')
GROUP BY `Country ISO3`
""")

hno_reach = hno_reach_spark.toPandas()
hno_reach["reach_ratio"] = pd.to_numeric(hno_reach["reach_ratio"], errors="coerce")
hno_reach["total_population"] = pd.to_numeric(hno_reach["total_population"], errors="coerce")

print("\n📊 Using predictions as primary data source (maximizing coverage)...")

# Use predictions table directly - COALESCE actual and predicted values
pin_predictions_spark = spark.sql("""
SELECT
  countryCode AS iso3,
  COALESCE(people_in_need, people_in_need_predicted) as people_in_need,
  is_predicted as pin_is_predicted
FROM workspace.unochadatasets.people_in_need_predictions
WHERE year = 2025
  AND COALESCE(people_in_need, people_in_need_predicted) IS NOT NULL
""")

hno_base = pin_predictions_spark.toPandas()
hno_base["people_in_need"] = pd.to_numeric(hno_base["people_in_need"], errors="coerce")
hno_base = hno_base.dropna(subset=["iso3", "people_in_need"])
hno_base = hno_base[hno_base["people_in_need"] > 0].copy()

# DEDUPLICATE: Keep first occurrence of each country (in case predictions table has duplicates)
hno_base = hno_base.drop_duplicates(subset=["iso3"], keep="first")

print(f"  - People in need data: {len(hno_base)} countries")
print(f"    · Actual: {(~hno_base['pin_is_predicted']).sum()}")
print(f"    · Predicted: {hno_base['pin_is_predicted'].sum()}")

# Apply demographic filtering if needed
if demographic_filter != "All" and len(demo_proportions) > 0:
    hno_base = hno_base.merge(demo_proportions, on="iso3", how="left")
    hno_base["demographic_proportion"] = hno_base["demographic_proportion"].fillna(1.0)
    hno_base["people_in_need"] = (hno_base["people_in_need"] * hno_base["demographic_proportion"]).round().astype('int64')
    hno_base = hno_base.drop(columns=["demographic_proportion"])

# Merge reach_ratio and total_population
hno_base = hno_base.merge(hno_reach[["iso3", "reach_ratio", "total_population"]], on="iso3", how="left")

# Use predictions table directly for requirements
req_predictions_spark = spark.sql("""
SELECT
  countryCode AS iso3,
  requirements_imputed as requirements,
  funding,
  imputation_flag as req_is_predicted
FROM workspace.unochadatasets.requirements_predictions
WHERE year = 2025
  AND requirements_imputed IS NOT NULL
  AND funding IS NOT NULL
""")

fts_base = req_predictions_spark.toPandas()
fts_base["requirements"] = pd.to_numeric(fts_base["requirements"], errors="coerce")
fts_base["funding"] = pd.to_numeric(fts_base["funding"], errors="coerce")
fts_base = fts_base.dropna(subset=["iso3", "requirements", "funding"])
fts_base = fts_base[fts_base["requirements"] > 0].copy()

# DEDUPLICATE: When there are duplicates, prefer actual data over predictions
# Sort by req_is_predicted (False first, then True) and keep first occurrence
fts_base = fts_base.sort_values("req_is_predicted").drop_duplicates(subset=["iso3"], keep="first")

print(f"  - Requirements data: {len(fts_base)} countries")
print(f"    · Actual: {(~fts_base['req_is_predicted']).sum()}")
print(f"    · Predicted: {fts_base['req_is_predicted'].sum()}")

# Temporal factor (optional)
if include_temporal == "Yes":
    fts_historical_spark = spark.sql("""
    SELECT
      `countryCode` AS iso3,
      `year`,
      SUM(`requirements`) AS requirements,
      SUM(`funding`) AS funding
    FROM workspace.unochadatasets.fts_requirements_funding_global
    WHERE `year` BETWEEN 2020 AND 2024
    GROUP BY `countryCode`, `year`
    """)
    
    fts_historical = fts_historical_spark.toPandas()
    fts_historical["requirements"] = pd.to_numeric(fts_historical["requirements"], errors="coerce")
    fts_historical["funding"] = pd.to_numeric(fts_historical["funding"], errors="coerce")
    fts_historical = fts_historical.dropna(subset=["iso3", "requirements", "funding"])
    fts_historical = fts_historical[fts_historical["requirements"] > 0].copy()
    
    fts_historical["coverage_ratio"] = (fts_historical["funding"] / fts_historical["requirements"]).clip(lower=0, upper=1)
    fts_historical["gap_score_year"] = fts_historical["requirements"] * (1 - fts_historical["coverage_ratio"])
    fts_historical["is_underfunded"] = (fts_historical["coverage_ratio"] < 1.0).astype(int)
    
    avg_gaps = fts_historical.groupby("iso3")["gap_score_year"].mean().reset_index()
    avg_gaps.columns = ["iso3", "avg_gap"]
    
    fts_historical_sorted = fts_historical.sort_values(["iso3", "year"])
    
    def count_consecutive_underfunded(group):
        group_sorted = group.sort_values("year", ascending=False)
        consecutive = 0
        for is_under in group_sorted["is_underfunded"]:
            if is_under == 1:
                consecutive += 1
            else:
                break
        return consecutive
    
    consecutive_years = fts_historical_sorted.groupby("iso3").apply(count_consecutive_underfunded).reset_index()
    consecutive_years.columns = ["iso3", "consecutive_years_underfunded"]
    
    temporal_factor = avg_gaps.merge(consecutive_years, on="iso3", how="left")
    temporal_factor["consecutive_years_underfunded"] = temporal_factor["consecutive_years_underfunded"].fillna(0).astype(int)
    
    LAMBDA = 0.2
    temporal_factor["temporal_factor_value"] = (
        temporal_factor["avg_gap"] + 
        LAMBDA * temporal_factor["consecutive_years_underfunded"] * temporal_factor["avg_gap"]
    )
    
    temporal_factor = temporal_factor[["iso3", "avg_gap", "consecutive_years_underfunded", "temporal_factor_value"]]
else:
    temporal_factor = pd.DataFrame(columns=["iso3", "avg_gap", "consecutive_years_underfunded", "temporal_factor_value"])

# Merge people_in_need and requirements (inner join - requires BOTH)
# This filter now applies to the prediction tables, treating predictions as valid data
df = hno_base.merge(fts_base, on="iso3", how="inner")
df = df.merge(temporal_factor, on="iso3", how="left")
df = df.merge(inform_severity_clean, on="iso3", how="left")
df = df.merge(complexity_clean, on="iso3", how="left")

df["avg_gap"] = df["avg_gap"].fillna(0).astype('float64')
df["consecutive_years_underfunded"] = df["consecutive_years_underfunded"].fillna(0).astype('int64')
df["temporal_factor_value"] = df["temporal_factor_value"].fillna(0).astype('float64')

df_country = df.copy()

print(f"\n✅ Combined dataset:")
print(f"  - Total countries: {len(df_country)}")
print(f"  - With people_in_need predictions: {df_country['pin_is_predicted'].sum()}")
print(f"  - With requirements predictions: {df_country['req_is_predicted'].sum()}")
print(f"  - With both predicted: {(df_country['pin_is_predicted'] & df_country['req_is_predicted']).sum()}")

# Calculate WMI components
df_country["coverage_ratio"] = df_country["funding"] / df_country["requirements"]
df_country["coverage_ratio"] = df_country["coverage_ratio"].clip(lower=0, upper=1)

# WMI Component 1: Severity (30%)
df_country["severity_component"] = (df_country["inform_severity_index"] * 10 * 0.3).fillna(0)

# WMI Component 2: Need Density (30%)
df_country["need_density_pct"] = (df_country["people_in_need"] / df_country["total_population"].replace(0, np.nan)) * 100
df_country["need_density_component"] = (df_country["need_density_pct"] * 0.3).fillna(0)

# WMI Component 3: Funding Gap (30%)
df_country["funding_gap_pct"] = (1 - df_country["coverage_ratio"]) * 100
df_country["funding_gap_component"] = df_country["funding_gap_pct"] * 0.3

# WMI Component 4: Complexity (10%)
df_country["complexity_component"] = (df_country["complexity_operating_environment"] * 10 * 0.1).fillna(0)

# WEIGHTED MISMATCH INDEX
df_country["overlooked_score"] = (
    df_country["severity_component"] +
    df_country["need_density_component"] +
    df_country["funding_gap_component"] +
    df_country["complexity_component"]
)

# Rank by overlooked_score (descending)
ranked = df_country.sort_values("overlooked_score", ascending=False).reset_index(drop=True)

final = ranked[[
    "iso3",
    "people_in_need",
    "requirements",
    "funding",
    "coverage_ratio",
    "reach_ratio",
    "total_population",
    "need_density_pct",
    "inform_severity_index",
    "complexity_operating_environment",
    "funding_gap_pct",
    "overlooked_score",
    "avg_gap",
    "consecutive_years_underfunded",
    "temporal_factor_value",
    "pin_is_predicted",
    "req_is_predicted"
]].copy()

# Ensure consistent data types
final["people_in_need"] = final["people_in_need"].astype('int64')
final["reach_ratio"] = final["reach_ratio"].astype('float64')
final["total_population"] = final["total_population"].astype('float64')
final["need_density_pct"] = final["need_density_pct"].astype('float64')
final["inform_severity_index"] = final["inform_severity_index"].astype('float64')
final["complexity_operating_environment"] = final["complexity_operating_environment"].astype('float64')
final["funding_gap_pct"] = final["funding_gap_pct"].astype('float64')
final["overlooked_score"] = final["overlooked_score"].astype('float64')
final["avg_gap"] = final["avg_gap"].astype('float64')
final["consecutive_years_underfunded"] = final["consecutive_years_underfunded"].astype('int64')
final["temporal_factor_value"] = final["temporal_factor_value"].astype('float64')
final["pin_is_predicted"] = final["pin_is_predicted"].astype('bool')
final["req_is_predicted"] = final["req_is_predicted"].astype('bool')

# Add country names
country_lookup_spark = spark.table("workspace.unochadatasets.country_iso3_lookup")
country_lookup = country_lookup_spark.toPandas()
final = final.merge(country_lookup, on="iso3", how="left")
final["country_name"] = final["country_name"].fillna(final["iso3"])

# Reorder columns
cols = ["iso3", "country_name"] + [col for col in final.columns if col not in ["iso3", "country_name"]]
final = final[cols]

# Count predictions used
pin_pred_count = final["pin_is_predicted"].sum()
req_pred_count = final["req_is_predicted"].sum()
both_pred_count = (final["pin_is_predicted"] & final["req_is_predicted"]).sum()
total_countries = len(final)

print(f"\n{'='*60}")
print(f"WMI Analysis Complete (predictions treated as valid data)")
print(f"  - Total countries: {total_countries}")
print(f"  - Crisis category: {crisis_category}")
print(f"  - Demographic filter: {demographic_filter}")
print(f"  - Temporal factor: {include_temporal}")
print(f"\n  📊 Predictions used:")
print(f"    · People in need: {pin_pred_count} / {total_countries} countries")
print(f"    · Requirements: {req_pred_count} / {total_countries} countries")
print(f"    · Both predicted: {both_pred_count} / {total_countries} countries")
print(f"{'='*60}")

# Save to Unity Catalog table with schema overwrite (handles schema changes)
schema = StructType([
    StructField("iso3", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("people_in_need", LongType(), True),
    StructField("requirements", DoubleType(), True),
    StructField("funding", DoubleType(), True),
    StructField("coverage_ratio", DoubleType(), True),
    StructField("reach_ratio", DoubleType(), True),
    StructField("total_population", DoubleType(), True),
    StructField("need_density_pct", DoubleType(), True),
    StructField("inform_severity_index", DoubleType(), True),
    StructField("complexity_operating_environment", DoubleType(), True),
    StructField("funding_gap_pct", DoubleType(), True),
    StructField("overlooked_score", DoubleType(), True),
    StructField("avg_gap", DoubleType(), True),
    StructField("consecutive_years_underfunded", IntegerType(), True),
    StructField("temporal_factor_value", DoubleType(), True),
    StructField("pin_is_predicted", BooleanType(), True),
    StructField("req_is_predicted", BooleanType(), True)
])

final_spark = spark.createDataFrame(final, schema=schema)
final_spark.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.unochadatasets.gap_rankings_2025")

print(f"✓ Rankings saved to workspace.unochadatasets.gap_rankings_2025")

In [0]:
# Export gap rankings for frontend development
import json

# Export to Unity Catalog Volume (serverless-compatible)
volume_path = "/Volumes/workspace/unochadatasets/exports"

# Export as CSV to Volume
csv_path = f"{volume_path}/gap_rankings_2025.csv"
final.to_csv(csv_path, index=False)
print(f"✓ CSV exported to Volume: {csv_path}")

# Export as JSON to Volume
json_path = f"{volume_path}/gap_rankings_2025.json"
final.to_json(json_path, orient="records", indent=2)
print(f"✓ JSON exported to Volume: {json_path}")

# FALLBACK: Also export to repo data directory for frontend
repo_data_path = "/Workspace/Repos/anicernak@gmail.com/UN-OCHA-/data/gap_rankings_2025.json"
final_json = final.to_dict(orient="records")
with open(repo_data_path, "w") as f:
    json.dump(final_json, f, indent=2)
print(f"✓ Fallback JSON exported to: {repo_data_path}")

print(f"\nTo download Volume files:")
print(f"1. Navigate to Catalog Explorer")
print(f"2. Go to workspace > unochadatasets > exports volume")
print(f"3. Right-click the file and select 'Download'")

In [0]:
# Display top 20 countries by gap score
display(final.head(50))

In [0]:
import pandas as pd
import numpy as np
import json
import time
from pathlib import Path

# Load INFORM Severity and Complexity data once using pandas
print("Loading INFORM severity and complexity data...")

# Read INFORM Severity CSV with pandas (skip title row)
inform_severity_pd = pd.read_csv("/Workspace/Users/maximilian.bach8@gmail.com/severity_csvs/INFORM_Severity_country.csv", skiprows=[0])
inform_severity_clean = inform_severity_pd[
    (inform_severity_pd["COUNTRY"].notna()) & 
    (inform_severity_pd["COUNTRY"] != "COUNTRY") &
    (inform_severity_pd["ISO3"] != "(a-z)")
][["ISO3", "INFORM Severity Index"]].copy()
inform_severity_clean.columns = ["iso3", "inform_severity_index"]
inform_severity_clean["inform_severity_index"] = pd.to_numeric(inform_severity_clean["inform_severity_index"], errors="coerce")
inform_severity_clean = inform_severity_clean.drop_duplicates(subset=["iso3"], keep="first")

# Read Complexity CSV with pandas
complexity_pd = pd.read_csv("/Workspace/Users/maximilian.bach8@gmail.com/severity_csvs/Complexity_of_the_crisis.csv", header=1, skiprows=[2,3])
complexity_pd = complexity_pd.rename(columns={"Unnamed: 4": "Iso3"})
complexity_clean = complexity_pd[
    (complexity_pd["Iso3"].notna()) & 
    (complexity_pd["Iso3"] != "Iso3") &
    (complexity_pd["Iso3"] != "")
][["Iso3", "Operating environment"]].copy()
complexity_clean.columns = ["iso3", "complexity_operating_environment"]
complexity_clean["complexity_operating_environment"] = pd.to_numeric(complexity_clean["complexity_operating_environment"], errors="coerce")
complexity_clean = complexity_clean.drop_duplicates(subset=["iso3"], keep="first")

print(f"  - INFORM severity: {len(inform_severity_clean)} countries")
print(f"  - Complexity: {len(complexity_clean)} countries\n")

# Get demographic filter from widget (apply to all combinations)
demographic_filter = dbutils.widgets.get("demographic_filter")

# Calculate demographic proportions once
if demographic_filter != "All":
    if demographic_filter == "Children":
        demo_filter = "Age_range IN ('0-4', '5-9', '10-14', '15-19')"
    elif demographic_filter == "Men":
        demo_filter = "Gender = 'm' AND Age_range NOT IN ('0-4', '5-9', '10-14', '15-19', 'all')"
    elif demographic_filter == "Women":
        demo_filter = "Gender = 'f' AND Age_range NOT IN ('0-4', '5-9', '10-14', '15-19', 'all')"
    
    demographic_pop = spark.sql(f"""
    SELECT 
        ISO3 as iso3,
        SUM(Population) as demographic_population
    FROM workspace.unochadatasets.cod_population_admin_0
    WHERE {demo_filter}
    GROUP BY ISO3
    """)
    
    total_pop = spark.sql("""
    SELECT 
        ISO3 as iso3,
        SUM(Population) as total_population
    FROM workspace.unochadatasets.cod_population_admin_0
    WHERE Age_range = 'all' AND Gender = 'all'
    GROUP BY ISO3
    """)
    
    demo_proportions_spark = demographic_pop.join(total_pop, "iso3", "inner")
    demo_proportions = demo_proportions_spark.toPandas()
    demo_proportions["demographic_proportion"] = (
        demo_proportions["demographic_population"] / demo_proportions["total_population"]
    )
    demo_proportions = demo_proportions[["iso3", "demographic_proportion"]]
else:
    demo_proportions = pd.DataFrame(columns=["iso3", "demographic_proportion"])

# Load country lookup once
country_lookup_spark = spark.table("workspace.unochadatasets.country_iso3_lookup")
country_lookup = country_lookup_spark.toPandas()

# All crisis categories
crisis_categories = [
    "ALL", "CCM", "CSS", "EDU", "ERY", "FSC", "HEA", "LOG", "MPC", "MS", 
    "NUT", "PRO", "PRO-CPN", "PRO-GBV", "PRO-HLP", "PRO-MIN", "SHL", "TEL", "WSH"
]

temporal_options = ["Yes", "No"]

output_dir = Path("/Workspace/Repos/anicernak@gmail.com/UN-OCHA-/data/rankings")
output_dir.mkdir(parents=True, exist_ok=True)

start_time = time.time()
total_combinations = len(crisis_categories) * len(temporal_options)
print(f"Generating WMI rankings for {total_combinations} parameter combinations...")
print(f"  - Crisis categories: {len(crisis_categories)}")
print(f"  - Temporal options: {len(temporal_options)}")
print(f"  - Demographic filter: {demographic_filter}")
print(f"  - Total files to generate: {total_combinations} JSON files\n")

successful = 0
failed = 0

for crisis_cat in crisis_categories:
    for temporal_opt in temporal_options:
        try:
            # Load HNO data with reach_ratio for this crisis category
            hno_reach_spark = spark.sql(f"""
            SELECT
              `Country ISO3` AS iso3,
              SUM(CAST(`Population` AS BIGINT)) AS total_population,
              (SUM(CAST(`Reached` AS DOUBLE)) / NULLIF(SUM(CAST(`Targeted` AS DOUBLE)), 0)) AS reach_ratio
            FROM workspace.unochadatasets.hpc_hno_2025
            WHERE `Cluster` = '{crisis_cat}'
              AND (`Category` IS NULL OR TRIM(`Category`) = '')
            GROUP BY `Country ISO3`
            """)
            
            hno_reach = hno_reach_spark.toPandas()
            hno_reach["reach_ratio"] = pd.to_numeric(hno_reach["reach_ratio"], errors="coerce")
            hno_reach["total_population"] = pd.to_numeric(hno_reach["total_population"], errors="coerce")
            
            # Load people_in_need predictions
            pin_predictions_spark = spark.sql("""
            SELECT
              countryCode AS iso3,
              COALESCE(people_in_need, people_in_need_predicted) as people_in_need,
              is_predicted as pin_is_predicted
            FROM workspace.unochadatasets.people_in_need_predictions
            WHERE year = 2025
              AND COALESCE(people_in_need, people_in_need_predicted) IS NOT NULL
            """)
            
            hno_base = pin_predictions_spark.toPandas()
            hno_base["people_in_need"] = pd.to_numeric(hno_base["people_in_need"], errors="coerce")
            hno_base = hno_base.dropna(subset=["iso3", "people_in_need"])
            hno_base = hno_base[hno_base["people_in_need"] > 0].copy()
            hno_base = hno_base.drop_duplicates(subset=["iso3"], keep="first")
            
            # Apply demographic filtering if needed
            if demographic_filter != "All" and len(demo_proportions) > 0:
                hno_base = hno_base.merge(demo_proportions, on="iso3", how="left")
                hno_base["demographic_proportion"] = hno_base["demographic_proportion"].fillna(1.0)
                hno_base["people_in_need"] = (hno_base["people_in_need"] * hno_base["demographic_proportion"]).round().astype('int64')
                hno_base = hno_base.drop(columns=["demographic_proportion"])
            
            # Merge reach_ratio and total_population
            hno_base = hno_base.merge(hno_reach[["iso3", "reach_ratio", "total_population"]], on="iso3", how="left")
            
            # Load requirements predictions
            req_predictions_spark = spark.sql("""
            SELECT
              countryCode AS iso3,
              requirements_imputed as requirements,
              funding,
              imputation_flag as req_is_predicted
            FROM workspace.unochadatasets.requirements_predictions
            WHERE year = 2025
              AND requirements_imputed IS NOT NULL
              AND funding IS NOT NULL
            """)
            
            fts_base = req_predictions_spark.toPandas()
            fts_base["requirements"] = pd.to_numeric(fts_base["requirements"], errors="coerce")
            fts_base["funding"] = pd.to_numeric(fts_base["funding"], errors="coerce")
            fts_base = fts_base.dropna(subset=["iso3", "requirements", "funding"])
            fts_base = fts_base[fts_base["requirements"] > 0].copy()
            fts_base = fts_base.sort_values("req_is_predicted").drop_duplicates(subset=["iso3"], keep="first")
            
            # Temporal factor (conditional)
            if temporal_opt == "Yes":
                fts_historical_spark = spark.sql("""
                SELECT
                  `countryCode` AS iso3,
                  `year`,
                  SUM(`requirements`) AS requirements,
                  SUM(`funding`) AS funding
                FROM workspace.unochadatasets.fts_requirements_funding_global
                WHERE `year` BETWEEN 2020 AND 2024
                GROUP BY `countryCode`, `year`
                """)
                
                fts_historical = fts_historical_spark.toPandas()
                fts_historical["requirements"] = pd.to_numeric(fts_historical["requirements"], errors="coerce")
                fts_historical["funding"] = pd.to_numeric(fts_historical["funding"], errors="coerce")
                fts_historical = fts_historical.dropna(subset=["iso3", "requirements", "funding"])
                fts_historical = fts_historical[fts_historical["requirements"] > 0].copy()
                
                fts_historical["coverage_ratio"] = (fts_historical["funding"] / fts_historical["requirements"]).clip(lower=0, upper=1)
                fts_historical["gap_score_year"] = fts_historical["requirements"] * (1 - fts_historical["coverage_ratio"])
                fts_historical["is_underfunded"] = (fts_historical["coverage_ratio"] < 1.0).astype(int)
                
                avg_gaps = fts_historical.groupby("iso3")["gap_score_year"].mean().reset_index()
                avg_gaps.columns = ["iso3", "avg_gap"]
                
                fts_historical_sorted = fts_historical.sort_values(["iso3", "year"])
                
                def count_consecutive_underfunded(group):
                    group_sorted = group.sort_values("year", ascending=False)
                    consecutive = 0
                    for is_under in group_sorted["is_underfunded"]:
                        if is_under == 1:
                            consecutive += 1
                        else:
                            break
                    return consecutive
                
                consecutive_years = fts_historical_sorted.groupby("iso3").apply(count_consecutive_underfunded).reset_index()
                consecutive_years.columns = ["iso3", "consecutive_years_underfunded"]
                
                temporal_factor = avg_gaps.merge(consecutive_years, on="iso3", how="left")
                temporal_factor["consecutive_years_underfunded"] = temporal_factor["consecutive_years_underfunded"].fillna(0).astype(int)
                
                LAMBDA = 0.2
                temporal_factor["temporal_factor_value"] = (
                    temporal_factor["avg_gap"] + 
                    LAMBDA * temporal_factor["consecutive_years_underfunded"] * temporal_factor["avg_gap"]
                )
                
                temporal_factor = temporal_factor[["iso3", "avg_gap", "consecutive_years_underfunded", "temporal_factor_value"]]
            else:
                temporal_factor = pd.DataFrame(columns=["iso3", "avg_gap", "consecutive_years_underfunded", "temporal_factor_value"])
            
            # Merge all data (inner join - requires BOTH people_in_need AND requirements)
            df = hno_base.merge(fts_base, on="iso3", how="inner")
            df = df.merge(temporal_factor, on="iso3", how="left")
            df = df.merge(inform_severity_clean, on="iso3", how="left")
            df = df.merge(complexity_clean, on="iso3", how="left")
            
            df["avg_gap"] = df["avg_gap"].fillna(0).astype('float64')
            df["consecutive_years_underfunded"] = df["consecutive_years_underfunded"].fillna(0).astype('int64')
            df["temporal_factor_value"] = df["temporal_factor_value"].fillna(0).astype('float64')
            
            df_country = df.copy()
            
            # Calculate WMI components
            df_country["coverage_ratio"] = df_country["funding"] / df_country["requirements"]
            df_country["coverage_ratio"] = df_country["coverage_ratio"].clip(lower=0, upper=1)
            
            # WMI Component 1: Severity (30%)
            df_country["severity_component"] = (df_country["inform_severity_index"] * 10 * 0.3).fillna(0)
            
            # WMI Component 2: Need Density (30%)
            df_country["need_density_pct"] = (df_country["people_in_need"] / df_country["total_population"].replace(0, np.nan)) * 100
            df_country["need_density_component"] = (df_country["need_density_pct"] * 0.3).fillna(0)
            
            # WMI Component 3: Funding Gap (30%)
            df_country["funding_gap_pct"] = (1 - df_country["coverage_ratio"]) * 100
            df_country["funding_gap_component"] = df_country["funding_gap_pct"] * 0.3
            
            # WMI Component 4: Complexity (10%)
            df_country["complexity_component"] = (df_country["complexity_operating_environment"] * 10 * 0.1).fillna(0)
            
            # WEIGHTED MISMATCH INDEX
            df_country["overlooked_score"] = (
                df_country["severity_component"] +
                df_country["need_density_component"] +
                df_country["funding_gap_component"] +
                df_country["complexity_component"]
            )
            
            # Rank by overlooked_score (descending)
            ranked = df_country.sort_values("overlooked_score", ascending=False).reset_index(drop=True)
            
            final = ranked[[
                "iso3",
                "people_in_need",
                "requirements",
                "funding",
                "coverage_ratio",
                "reach_ratio",
                "total_population",
                "need_density_pct",
                "inform_severity_index",
                "complexity_operating_environment",
                "funding_gap_pct",
                "overlooked_score",
                "avg_gap",
                "consecutive_years_underfunded",
                "temporal_factor_value",
                "pin_is_predicted",
                "req_is_predicted"
            ]].copy()
            
            # Ensure consistent data types
            final["people_in_need"] = final["people_in_need"].astype('int64')
            final["reach_ratio"] = final["reach_ratio"].astype('float64')
            final["total_population"] = final["total_population"].astype('float64')
            final["need_density_pct"] = final["need_density_pct"].astype('float64')
            final["inform_severity_index"] = final["inform_severity_index"].astype('float64')
            final["complexity_operating_environment"] = final["complexity_operating_environment"].astype('float64')
            final["funding_gap_pct"] = final["funding_gap_pct"].astype('float64')
            final["overlooked_score"] = final["overlooked_score"].astype('float64')
            final["avg_gap"] = final["avg_gap"].astype('float64')
            final["consecutive_years_underfunded"] = final["consecutive_years_underfunded"].astype('int64')
            final["temporal_factor_value"] = final["temporal_factor_value"].astype('float64')
            final["pin_is_predicted"] = final["pin_is_predicted"].astype('bool')
            final["req_is_predicted"] = final["req_is_predicted"].astype('bool')
            
            # Add country names
            final = final.merge(country_lookup, on="iso3", how="left")
            final["country_name"] = final["country_name"].fillna(final["iso3"])
            
            # Reorder columns
            cols = ["iso3", "country_name"] + [col for col in final.columns if col not in ["iso3", "country_name"]]
            final = final[cols]
            
            # Save to JSON file
            output_filename = f"{crisis_cat}_{temporal_opt}.json"
            output_path = output_dir / output_filename
            
            final_json = final.to_dict(orient="records")
            with open(output_path, "w") as f:
                json.dump(final_json, f, indent=2)
            
            successful += 1
            print(f"✓ [{successful}/{total_combinations}] Generated: {output_filename} ({len(final)} countries)")
            
        except Exception as e:
            failed += 1
            print(f"✗ [{successful+failed}/{total_combinations}] Failed: {crisis_cat}_{temporal_opt} - {str(e)}")

end_time = time.time()
elapsed = end_time - start_time

print(f"\n{'='*80}")
print(f"Batch generation complete!")
print(f"  - Total combinations: {total_combinations}")
print(f"  - Successful: {successful}")
print(f"  - Failed: {failed}")
print(f"  - Time elapsed: {elapsed:.1f} seconds")
print(f"  - Output directory: {output_dir}")
print(f"{'='*80}")